# 03. nn.py: собираем кирпичики DiT

От линейной проекции до RoPE и адаптивной нормализации. Разбираем все классы nn.py на уменьшенных размерах, сохраняя порядок операций.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/03_nn_building_blocks.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


### 1. Загружаем классы без CUDA compilation

Учебный adapter читает неизменённый файл, убирает декораторы compile/CUDA autocast в отдельном модуле и выбирает FP32. Глобальный torch не меняется. Это позволяет учиться на CPU/MPS; преобразования перечислены в README.


In [ ]:
from labkit.native import load_dit_class
load_dit_class(torch.float32)
import sys
nn_ref=sys.modules['_k5_lab_float32_128.nn']


### 2. VisualEmbeddings: патч — маленький кубик

Блок 1×2×2 объединяет четыре соседних латентных пикселя. Их каналы склеиваются, затем Linear переводит их в model_dim. В Lite T2V вход содержит 16 noisy + 16 conditioning + 1 mask=33 канала; Linear принимает 132 числа на патч.


In [ ]:
visual=torch.arange(1*4*4*16,dtype=torch.float32).reshape(1,4,4,16)
layer=nn_ref.VisualEmbeddings(16,48,(1,2,2))
embedding=layer(visual)
print(visual.shape,'→',embedding.shape,'Linear input:',layer.in_layer.in_features)


### 3. TextEmbeddings и TimeEmbeddings

Проекция текста делает ширину Qwen совместимой с DiT. Время сначала преобразуется в sin/cos разных частот, затем в MLP. CLIP pooled проходит собственную проекцию и складывается с time embedding. Это conditioning нормализаций, а не дополнительный visual token.


In [ ]:
text_proj=nn_ref.TextEmbeddings(32,48); clip_proj=nn_ref.TextEmbeddings(16,32)
time_proj=nn_ref.TimeEmbeddings(48,32)
text=torch.randn(1,6,32); pooled=torch.randn(1,16)
time=time_proj(torch.tensor([500.]))+clip_proj(pooled)
print('text:',text_proj(text).shape,'time+CLIP:',time.shape)


### 4. RoPE1D: поворот вместо прибавления позиции

RoPE вращает пары координат Query/Key. Одновременный поворот сохраняет скалярное произведение, разные повороты кодируют относительное положение. В коде последняя ось разбивается на пары. Пример с 2D объясняет механику без большой матрицы.


In [ ]:
angle=torch.tensor(0.6)
R=torch.stack([torch.stack([angle.cos(),-angle.sin()]),torch.stack([angle.sin(),angle.cos()])])
a=torch.tensor([1.,2.]); b=torch.tensor([3.,-1.])
print('dot before:',a@b,'after:',(R@a)@(R@b))
rope=nn_ref.RoPE1D(12)(torch.arange(6))
print('RoPE1D:',rope.shape)


### 5. RoPE3D: три координаты

Части head_dim отвечают времени, высоте и ширине. Для Lite axes_dims=(16,24,24), head_dim=64, model_dim/head_dim=28 heads. scale_factor=(1,2,2) масштабирует позиционные частоты, а не размеры тензора.


In [ ]:
rope3=nn_ref.RoPE3D((4,4,4))
pos=[torch.arange(1),torch.arange(2),torch.arange(2)]
print('3D rotations:',rope3((1,2,2),pos,(1,2,2)).shape)


### 6. Modulation: время меняет обработку признаков

MLP выдаёт shift, scale и gate. После norm вычисляется norm(x)·(1+scale)+shift; residual добавляет gate·out. Нулевая инициализация modulation даёт gate=0 и сохраняет исходный residual на старте обучения.


In [ ]:
mod=nn_ref.Modulation(32,48,3)
shift,scale,gate=mod(time).chunk(3,-1)
x=torch.randn(1,4,48); norm=torch.nn.LayerNorm(48,elementwise_affine=False)
z=nn_ref.apply_scale_shift_norm(norm,x,scale,shift)
result=nn_ref.apply_gate_sum(x,z,gate)
print('gate max:',gate.abs().max().item(),'residual change:',(result-x).abs().max().item())


### 7. Три attention-класса

MultiheadSelfAttentionEnc обрабатывает text tokens и padding. MultiheadSelfAttentionDec обрабатывает visual tokens и может выбирать NABLA. MultiheadCrossAttention берёт Q из visual, K/V из text. Все используют Q/K RMSNorm. Код ниже проверяет формы; веса случайные, выход не несёт семантики.


In [ ]:
enc=nn_ref.MultiheadSelfAttentionEnc(48,12,'sdpa',True)
dec=nn_ref.MultiheadSelfAttentionDec(48,12,'sdpa')
cross=nn_ref.MultiheadCrossAttention(48,12,'sdpa',True)
text48=text_proj(text); vis48=embedding.reshape(1,4,48)
print('text self:',enc(text48,rope,torch.ones(1,6,dtype=torch.bool)).shape)
visual_rope=rope3((1,2,2),pos).reshape(4,1,6,2,2)
print('visual self:',dec(vis48,visual_rope).shape)
print('cross:',cross(vis48,text48).shape)


### 8. FeedForward и OutLayer

FeedForward расширяет размерность, применяет GELU и сжимает обратно. OutLayer использует modulation, norm и Linear, затем разворачивает патчи в исходную решётку. Предсказываются 16 каналов velocity, а не RGB.


In [ ]:
ff=nn_ref.FeedForward(48,96); out=nn_ref.OutLayer(48,32,16,(1,2,2))
print('FF:',ff(vis48).shape)
print('unpatchified velocity:',out(embedding,text48,time).shape)


#


In [ ]:
#


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
